# Punto 7 - Procesamiento analítico descriptivo con PySpark

Este notebook realiza procesamiento analítico descriptivo usando PySpark sobre los archivos limpios del proyecto.

Archivos de entrada esperados:

- `data/trusted_places_clean.csv`
- `data/trusted_place_types_clean.csv`
- `data/trusted_place_hours_clean.csv`

Los archivos representan la salida de la zona `trusted/` del Data Lake. El objetivo es generar resultados analíticos por barrio, tipo de lugar, disponibilidad en fin de semana y nivel de precio.

## 1. Importación de librerías y creación de SparkSession

Esta sección importa PySpark, configura rutas locales del proyecto y crea una sesión Spark en modo local.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    col,
    count,
    desc,
    first,
    round,
    sum,
    when
)

In [2]:
spark = (
    SparkSession.builder
    .appName("Proyecto2Punto7")
    .master("local[*]")
    .getOrCreate()
)

spark

## 2. Configuración de rutas

El notebook debe ejecutarse desde la carpeta raíz `punto7-pyspark` o desde la carpeta `notebooks`.

La siguiente celda detecta automáticamente la ubicación correcta del proyecto.

In [3]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

PLACES_PATH = DATA_DIR / "trusted_places_clean.csv"
TYPES_PATH = DATA_DIR / "trusted_place_types_clean.csv"
HOURS_PATH = DATA_DIR / "trusted_place_hours_clean.csv"

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PLACES_PATH exists:", PLACES_PATH.exists())
print("TYPES_PATH exists:", TYPES_PATH.exists())
print("HOURS_PATH exists:", HOURS_PATH.exists())

BASE_DIR: d:\code\big-data\Proyecto2-BigData\punto7-pyspark
DATA_DIR: d:\code\big-data\Proyecto2-BigData\punto7-pyspark\data
OUTPUT_DIR: d:\code\big-data\Proyecto2-BigData\punto7-pyspark\outputs
PLACES_PATH exists: True
TYPES_PATH exists: True
HOURS_PATH exists: True


## 3. Lectura de archivos trusted

Se cargan las tres tablas principales usando PySpark.

In [4]:
places_df = spark.read.csv(str(PLACES_PATH), header=True, inferSchema=True)
types_df = spark.read.csv(str(TYPES_PATH), header=True, inferSchema=True)
hours_df = spark.read.csv(str(HOURS_PATH), header=True, inferSchema=True)

## 4. Exploración inicial de esquemas y registros

Estas salidas sirven como evidencia de lectura correcta de los archivos y detección de tipos.

In [5]:
print("=== Schema places ===")
places_df.printSchema()

print("=== Schema types ===")
types_df.printSchema()

print("=== Schema hours ===")
hours_df.printSchema()

=== Schema places ===
root
 |-- place_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lng: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- price_level: integer (nullable = true)
 |-- review_count: integer (nullable = true)

=== Schema types ===
root
 |-- place_id: string (nullable = true)
 |-- type: string (nullable = true)

=== Schema hours ===
root
 |-- place_id: string (nullable = true)
 |-- day_es: string (nullable = true)
 |-- day_en: string (nullable = true)
 |-- is_open: boolean (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- close_time: string (nullable = true)
 |-- raw: string (nullable = true)



In [6]:
print("=== Places sample ===")
places_df.show(5, truncate=False)

print("=== Types sample ===")
types_df.show(5, truncate=False)

print("=== Hours sample ===")
hours_df.show(5, truncate=False)

=== Places sample ===
+---------------------------+---------------------------+---------------------------------------------------------------------------------+------------+-----------------+------------------+------+-----------+------------+
|place_id                   |name                       |address                                                                          |neighborhood|lat              |lng               |rating|price_level|review_count|
+---------------------------+---------------------------+---------------------------------------------------------------------------------+------------+-----------------+------------------+------+-----------+------------+
|ChIJ-1IbBmGCRo4RpY5njdWV_0M|Arte Alto | Galería de arte|Cl. 9 #43 c - 50, El Poblado, Medellín, El Poblado, Medellín, Antioquia, Colombia|El Poblado  |6.210606299999999|-75.5726244       |5.0   |NULL       |5           |
|ChIJ-5iJ83iCRo4RfDv9AusSg8Y|Restaurante Donde La Suegra|Guayabal, Medellín, Antioquia, Co

In [7]:
print("Places records:", places_df.count())
print("Types records:", types_df.count())
print("Hours records:", hours_df.count())

Places records: 969
Types records: 1573
Hours records: 5999


## 5. Preparación analítica

Se crean tablas derivadas para relacionar lugares con su tipo principal y con información de apertura en fin de semana.

In [8]:
primary_types_df = (
    types_df
    .groupBy("place_id")
    .agg(
        first("type", ignorenulls=True).alias("primary_type"),
        count("type").alias("types_count")
    )
)

primary_types_detail_df = (
    primary_types_df
    .join(
        places_df.select("place_id", "name", "neighborhood"),
        on="place_id",
        how="left"
    )
    .select("place_id", "name", "neighborhood", "primary_type", "types_count")
)

primary_types_detail_df.show(10, truncate=False)

+---------------------------+---------------------------------------+------------+------------+-----------+
|place_id                   |name                                   |neighborhood|primary_type|types_count|
+---------------------------+---------------------------------------+------------+------------+-----------+
|ChIJ-1IbBmGCRo4RpY5njdWV_0M|Arte Alto | Galería de arte            |El Poblado  |art_gallery |1          |
|ChIJ-5U1CTWDRo4Rq225QIQCdTU|Sazón Leña                             |El Poblado  |restaurant  |1          |
|ChIJ-5iJ83iCRo4RfDv9AusSg8Y|Restaurante Donde La Suegra            |Medellín    |restaurant  |1          |
|ChIJ-63vYOUpRI4RUSOyAzRl9X8|Lollipop club                          |El Poblado  |night_club  |1          |
|ChIJ-6ntz-qCRo4RdoGDksNP9VM|Parque Roa                             |Medellín    |park        |1          |
|ChIJ-8XP5ikoRI4RbGXZfyvahi4|37 Park Medellín | Restaurante & Bar   |El Poblado  |bar         |2          |
|ChIJ-TnstaGDRo4RFVjrMySUyzM

In [9]:
hours_prepared_df = (
    hours_df
    .withColumn(
        "is_open_bool",
        when(col("is_open").cast("string").isin("true", "True", "1", "yes", "YES"), True)
        .otherwise(False)
    )
)

weekend_open_df = (
    hours_prepared_df
    .filter((col("day_en").isin("Saturday", "Sunday")) & (col("is_open_bool") == True))
    .groupBy("place_id")
    .agg(count("day_en").alias("weekend_open_days"))
    .withColumn("opens_on_weekend", col("weekend_open_days") > 0)
)

weekend_open_detail_df = (
    weekend_open_df
    .join(
        places_df.select("place_id", "name", "neighborhood", "rating", "review_count"),
        on="place_id",
        how="left"
    )
    .select("place_id", "name", "neighborhood", "rating", "review_count", "weekend_open_days", "opens_on_weekend")
)

weekend_open_detail_df.show(10, truncate=False)

+---------------------------+--------------------------------------------------+---------------------------------------+------+------------+-----------------+----------------+
|place_id                   |name                                              |neighborhood                           |rating|review_count|weekend_open_days|opens_on_weekend|
+---------------------------+--------------------------------------------------+---------------------------------------+------+------------+-----------------+----------------+
|ChIJ937qbZqCRo4Rk0fvYaMFzgM|Nespresso - CC. El Tesoro                         |Loma El Tesoro con Transversal Superior|4.5   |5           |2                |true            |
|ChIJITZr2VGCRo4R9I3DYF1mu2M|Mall La Bota del Día                              |Zona 9                                 |4.2   |5           |2                |true            |
|ChIJXWT1geKDRo4Rudo38tvq1Os|Falero El Uruguayo, choripanes, parrilla y amigos.|Zona 2                                 |

In [10]:
places_enriched_df = (
    places_df
    .join(primary_types_df, on="place_id", how="left")
    .join(weekend_open_df, on="place_id", how="left")
    .fillna({
        "primary_type": "Sin categoría",
        "types_count": 0,
        "weekend_open_days": 0,
        "opens_on_weekend": False,
        "price_level": 0,
        "review_count": 0
    })
)

places_enriched_df.select(
    "place_id",
    "name",
    "neighborhood",
    "primary_type",
    "rating",
    "price_level",
    "review_count",
    "weekend_open_days",
    "opens_on_weekend"
).show(10, truncate=False)

+---------------------------+------------------------------------+--------------------------+------------+------+-----------+------------+-----------------+----------------+
|place_id                   |name                                |neighborhood              |primary_type|rating|price_level|review_count|weekend_open_days|opens_on_weekend|
+---------------------------+------------------------------------+--------------------------+------------+------+-----------+------------+-----------------+----------------+
|ChIJ-1IbBmGCRo4RpY5njdWV_0M|Arte Alto | Galería de arte         |El Poblado                |art_gallery |5.0   |0          |5           |0                |false           |
|ChIJ-5iJ83iCRo4RfDv9AusSg8Y|Restaurante Donde La Suegra         |Medellín                  |restaurant  |5.0   |0          |3           |0                |false           |
|ChIJ-5U1CTWDRo4Rq225QIQCdTU|Sazón Leña                          |El Poblado                |restaurant  |1.8   |0          |5    

## 5.1 Muestra enriquecida orientada a lectura

Se muestra una vista reducida de lugares con nombre, barrio, tipo, rating, reseñas y disponibilidad en fin de semana. Esta salida facilita la lectura de resultados sin depender únicamente de identificadores técnicos.

In [11]:
readable_places_sample_df = (
    places_enriched_df
    .select(
        "name",
        "neighborhood",
        "primary_type",
        "rating",
        "price_level",
        "review_count",
        "weekend_open_days",
        "opens_on_weekend"
    )
    .orderBy(desc("rating"), desc("review_count"))
)

readable_places_sample_df.show(15, truncate=False)

+---------------------------------------+------------------------+------------+------+-----------+------------+-----------------+----------------+
|name                                   |neighborhood            |primary_type|rating|price_level|review_count|weekend_open_days|opens_on_weekend|
+---------------------------------------+------------------------+------------+------+-----------+------------+-----------------+----------------+
|Lucerito                               |El Poblado              |cafe        |5.0   |0          |5           |1                |true            |
|Terra Maaru (café restaurante natural) |El Poblado              |restaurant  |5.0   |0          |5           |1                |true            |
|SILO Cocina                            |Cl. 2 Sur #25 115 Piso 6|restaurant  |5.0   |0          |5           |2                |true            |
|Casa Loop                              |El Poblado              |bakery      |5.0   |0          |5           |2      

## 6. Análisis descriptivo por barrio

Responde la pregunta: ¿cuáles barrios concentran mayor cantidad de lugares y cuál es su rating promedio?

In [12]:
places_by_neighborhood_df = (
    places_enriched_df
    .groupBy("neighborhood")
    .agg(
        count("place_id").alias("places_count"),
        round(avg("rating"), 2).alias("average_rating"),
        sum("review_count").alias("total_reviews")
    )
    .orderBy(desc("places_count"))
)

places_by_neighborhood_df.show(15, truncate=False)

+-------------+------------+--------------+-------------+
|neighborhood |places_count|average_rating|total_reviews|
+-------------+------------+--------------+-------------+
|El Poblado   |567         |4.37          |2650         |
|Guayabal     |47          |4.49          |201          |
|Zona 2       |44          |4.47          |214          |
|Zona 9       |30          |4.6           |135          |
|Medellín     |25          |4.42          |111          |
|La Abadia    |16          |4.25          |73           |
|San Fernando |9           |4.21          |42           |
|Casaloma     |6           |4.63          |27           |
|Envigado     |6           |3.95          |30           |
|Poblado      |4           |4.7           |17           |
|Los Cristales|4           |4.43          |19           |
|Zona 1       |4           |4.35          |20           |
|Montiel II   |3           |4.07          |12           |
|La Orquidea  |3           |4.87          |12           |
|NULL         

## 7. Análisis descriptivo por tipo de lugar

Responde la pregunta: ¿qué categorías concentran más lugares y mayor volumen de reseñas?

In [13]:
places_by_type_df = (
    places_enriched_df
    .groupBy("primary_type")
    .agg(
        count("place_id").alias("places_count"),
        round(avg("rating"), 2).alias("average_rating"),
        sum("review_count").alias("total_reviews")
    )
    .orderBy(desc("places_count"))
)

places_by_type_df.show(15, truncate=False)

+----------------------+------------+--------------+-------------+
|primary_type          |places_count|average_rating|total_reviews|
+----------------------+------------+--------------+-------------+
|restaurant            |342         |4.28          |1598         |
|bar                   |236         |4.54          |1119         |
|cafe                  |173         |4.4           |756          |
|night_club            |71          |4.05          |324          |
|bakery                |51          |4.38          |251          |
|park                  |39          |4.59          |185          |
|art_gallery           |26          |4.74          |115          |
|meal_delivery         |14          |3.95          |70           |
|museum                |6           |4.58          |27           |
|lodging               |3           |4.47          |15           |
|meal_takeaway         |3           |4.6           |12           |
|clothing_store        |1           |4.3           |5         

## 8. Análisis de apertura en fin de semana

Permite identificar cuántos lugares tienen disponibilidad sábado o domingo por tipo de lugar.

In [14]:
weekend_coverage_df = (
    places_enriched_df
    .groupBy("primary_type")
    .agg(
        count("place_id").alias("places_count"),
        sum(when(col("opens_on_weekend") == True, 1).otherwise(0)).alias("weekend_open_places")
    )
    .withColumn(
        "weekend_open_ratio",
        round((col("weekend_open_places") / col("places_count")) * 100, 2)
    )
    .orderBy(desc("weekend_open_places"))
)

weekend_coverage_df.show(15, truncate=False)

+----------------------+------------+-------------------+------------------+
|primary_type          |places_count|weekend_open_places|weekend_open_ratio|
+----------------------+------------+-------------------+------------------+
|restaurant            |342         |309                |90.35             |
|bar                   |236         |209                |88.56             |
|cafe                  |173         |135                |78.03             |
|night_club            |71          |59                 |83.1              |
|bakery                |51          |49                 |96.08             |
|park                  |39          |29                 |74.36             |
|art_gallery           |26          |19                 |73.08             |
|meal_delivery         |14          |14                 |100.0             |
|museum                |6           |5                  |83.33             |
|meal_takeaway         |3           |3                  |100.0             |

## 9. Análisis de rating por nivel de precio

Permite observar si el nivel de precio se relaciona con el rating promedio.

In [15]:
price_rating_analysis_df = (
    places_enriched_df
    .filter(col("price_level") > 0)
    .groupBy("price_level")
    .agg(
        count("place_id").alias("places_count"),
        round(avg("rating"), 2).alias("average_rating"),
        sum("review_count").alias("total_reviews")
    )
    .orderBy("price_level")
)

price_rating_analysis_df.show(truncate=False)

+-----------+------------+--------------+-------------+
|price_level|places_count|average_rating|total_reviews|
+-----------+------------+--------------+-------------+
|1          |17          |4.42          |81           |
|2          |203         |4.3           |1009         |
|3          |24          |4.52          |120          |
|4          |4           |4.43          |20           |
+-----------+------------+--------------+-------------+



## 10. Exportación de resultados

Los resultados analíticos se exportan a CSV en la carpeta `outputs/`. Además de las agregaciones, se genera una salida legible con nombres de lugares para facilitar la revisión de resultados.

In [17]:
places_by_neighborhood_df.toPandas().to_csv(
    OUTPUT_DIR / "places_by_neighborhood.csv",
    index=False
)

places_by_type_df.toPandas().to_csv(
    OUTPUT_DIR / "places_by_type.csv",
    index=False
)

weekend_coverage_df.toPandas().to_csv(
    OUTPUT_DIR / "weekend_coverage.csv",
    index=False
)

price_rating_analysis_df.toPandas().to_csv(
    OUTPUT_DIR / "price_rating_analysis.csv",
    index=False
)

readable_places_sample_df.limit(50).toPandas().to_csv(
    OUTPUT_DIR / "top_places_readable.csv",
    index=False
)

print("Archivos generados:")
for file in OUTPUT_DIR.glob("*.csv"):
    print("-", file)

Archivos generados:
- d:\code\big-data\Proyecto2-BigData\punto7-pyspark\outputs\places_by_neighborhood.csv
- d:\code\big-data\Proyecto2-BigData\punto7-pyspark\outputs\places_by_type.csv
- d:\code\big-data\Proyecto2-BigData\punto7-pyspark\outputs\price_rating_analysis.csv
- d:\code\big-data\Proyecto2-BigData\punto7-pyspark\outputs\top_places_readable.csv
- d:\code\big-data\Proyecto2-BigData\punto7-pyspark\outputs\weekend_coverage.csv


## 11. Cierre de sesión Spark

Se finaliza la sesión Spark al terminar el procesamiento.

In [18]:
spark.stop()